# 🧹 Data Preprocessing - Final Pipeline

In this section, we will **consolidate everything** we’ve done so far into one final script using Scikit-Learn pipelines. This includes:

1. Creating a stratified test set
2. Handling missing values
3. Encoding categorical variables
4. Scaling numerical features
5. Combining everything using <mark>`Pipeline`</mark> and <mark>`ColumnTransformer`</mark>

This will ensure **clean, modular, and reproducible code** — perfect for production and education.

---

## 🚀 Final Preprocessing Code using Scikit-Learn Pipelines

```python
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# from sklearn.preprocessing import OrdinalEncoder  # Uncomment if you prefer ordinal
```

### 1. Load the Data

```python
housing = pd.read_csv("housing.csv")
```

### 2. Create a Stratified Test Set Based on Income Category

```python
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)
```

```python
split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = housing.loc[test_index].drop("income_cat", axis=1)
```

### 3. Work on a Copy of Training Data

```python
housing = strat_train_set.copy()
```

### 4. Separate Predictors and Labels

```python
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)
```

### 5. Separate Numerical and Categorical Columns

```python
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]
```

### 6. Pipelines

#### Numerical Pipeline

```python
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
```

#### Categorical Pipeline

```python
cat_pipeline = Pipeline([
    # ("ordinal", OrdinalEncoder())  # Use this if you prefer ordinal encoding
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
```

#### Full Pipeline

```python
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])
```

### 7. Transform the Data

```python
housing_prepared = full_pipeline.fit_transform(housing)
```

```python
# housing_prepared is now a NumPy array ready for training
print(housing_prepared.shape)
```
